Homework 1 [100 points]
=======

### Deliverables:

Submit your queries (and only those) using the `submission_template.txt` file that is posted on Canvas. Follow the instructions on the file! Upload the file at Canvas.


### Instructions / Notes:

* You **may** create new IPython notebook cells to use for e.g. testing, debugging, exploring, etc.- this is encouraged in fact!- **just make sure that your final answer for each question is _in its own cell_ and _clearly indicated_**
* When you see `In [*]:` to the left of the cell you are executing, this means that the code / query is _running_.
    * **If the cell is hanging- i.e. running for too long: To restart the SQL connection, you must restart the entire python kernel**
    * To restart kernel using the menu bar: "Kernel >> Restart >> Clear all outputs & restart"), then re-execute the sql connection cell at top
    * You will also need to restart the connection if you want to load a different version of the database file
* Remember:
    * `%sql [SQL]` is for _single line_ SQL queries
    * `%%sql [SQL]` is for _multi line_ SQL queries
* _Have fun!_

Section 1: Relational Algebra [25 points]
=======

Problem 1: Relational Algebra [25 points]
---------

Consider the following relational schema for conference publications:
*  `Article(artid, title, confid, numpages)`
*  `Conference(confid, name, year, location)`
*  `Author(artid, pid)`
*  `Person(pid, name, affiliation)`

Express the following queries in the extended Relational Algebra (you can also use the aggregation operator if necessary). To write the RA expression, use the LaTex mode that ipython notebook provides. For example:

$$\pi_{name}(\sigma_{affiliation="UW-Madison"}(Person))$$ 

### Part (a) [8 points]

Output the name of every person affiliated with `UW-Madison` who has published an article in a 2021 conference.

$$\pi_{name}\Bigl(\sigma_{affiliation="UW-Madison"}(Person) \bowtie Author \bowtie \bigl(Article \bowtie \sigma_{year=2021}(Conference)\bigr)\Bigr)$$

### Part (b) [9 points]

Output the names of the people who coauthored an article with `John Doe`. Be careful: a person cannot be coauthor with herself!

$$\pi_{name}(\sigma_{name \neq "John Doe"}(Person \Join Author \Join (\pi_{artid}(\sigma_{name = "John Doe"}(Person \Join Author)))))$$

### Part (c) [8 points]

Translate the following SQL query to Relational Algebra.

In [2]:
%%sql
SELECT pid, COUNT(A.artid)
FROM Article A, Conference C, Author U
WHERE A.confid = C.confid AND C.name = "PODS" AND U.artid = A.artid
GROUP BY pid ;

 * sqlite:///hw1.db
(sqlite3.OperationalError) no such table: Article
[SQL: SELECT pid, COUNT(A.artid)
FROM Article A, Conference C, Author U
WHERE A.confid = C.confid AND C.name = "PODS" AND U.artid = A.artid
GROUP BY pid ;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


$$
\gamma_{pid, COUNT(artid)}((\sigma_{name = "PODS"}(Conference)) \Join Article \Join Author)
$$

Section 2: SQL [75 points]
=======

Run the cell below to load the database `hw1.db` (make sure the database file, `hw1.db`, is in the same directory as this IPython notebook is running in)

Some of the problems involve _changing_ this database (e.g. deleting rows)- you can always re-download `hw1.db` or make a copy if you want to start fresh!

In [3]:
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%sql sqlite:///hw1.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Problem 2: Linear Algebra [25 points]
------------------------

Two random 3x3 ($N=3$) matrices have been provided in tables `A` and `B`, having the following schema:
> * `i INT`:   Row index
> * `j INT`:   Column index
> * `val INT`: Cell value

**Note: all of your answers below _must_ work for any _square_ matrix sizes, i.e. any value of $N$**.

Note how the matrices are represented - why do we choose this format?  Run the following queries to see the matrices in a nice format:

In [4]:
%sql SELECT group_concat(val, ' , ') AS "A" FROM A GROUP BY i;

 * sqlite:///hw1.db
Done.


A
"7 , 5 , 8"
"10 , 7 , 7"
"2 , 0 , 5"


In [5]:
%sql SELECT group_concat(val, ' , ') AS "B" FROM B GROUP BY i;

 * sqlite:///hw1.db
Done.


B
"9 , 6 , 10"
"7 , 6 , 9"
"1 , 1 , 7"


### Part (a): Matrix addition [5 points]

The sum of a matrix $A$ (having dimensions $n\times m$) and a matrix $B$ (having dimensions $n\times m$) is the matrix $C$ (of dimension $n\times m$) having cell at row $i$ and column $j$ equal to:

$C_{ij} = A_{ij} + B_{ij}$

Write a single SQL query to get the sum of $A$ and $B$ (in the same format as $A$ and $B$):

In [ ]:
%sql SELECT group_concat(val, ' , ') AS "C" FROM (SELECT A.i, A.j, A.val + B.val AS "VAL" FROM A JOIN B WHERE A.i = B.i AND A.j = B.j) GROUP BY i;

 * sqlite:///hw1.db
Done.


C
"16 , 11 , 18"
"17 , 13 , 16"
"3 , 1 , 12"


### Part (b): Dot product [5 points]

The _dot product_ of two vectors

$a = \begin{bmatrix}a_1 & a_2 & \dots & a_n\end{bmatrix}$

and

$b = \begin{bmatrix}b_1 & b_2 & \dots & b_n\end{bmatrix}$

is

$a\cdot b = \sum_{i=1}^n a_ib_i = a_1b_1 + a_2b_2 + \dots + a_nb_n$

Write a _single SQL query_ to take the dot product of the **second column of $A$** and the **third column of $B$.**:

In [ ]:
%sql SELECT SUM(A.val * B.val) AS "VAL" FROM A JOIN B ON A.i = B.i WHERE A.j = 1 AND B.j = 2;

 * sqlite:///hw1.db
Done.


VAL
113


### Part (c): Matrix multiplication [7 points]

The product of a matrix $A$ (having dimensions $n\times m$) and a matrix $B$ (having dimensions $m\times p$) is the matrix $C$ (of dimension $n\times p$) having cell at row $i$ and column $j$ equal to:

$C_{ij} = \sum_{k=1}^m A_{ik}B_{kj}$

In other words, to multiply two matrices, get each cell of the resulting matrix $C$, $C_{ij}$, by taking the _dot product_ of the $i$th row of $A$ and the $j$th column of $B$.

Write a single SQL query to get the matrix product of $A$ and $B$ (in the same format as $A$ and $B$):

In [8]:
%%sql
SELECT group_concat(result_val, ' , ') AS "C" FROM
(SELECT
    A.i AS result_row,
    B.j AS result_col,
    SUM(A.val * B.val) AS result_val
FROM A JOIN B ON A.j = B.i
GROUP BY A.i, B.j
ORDER BY result_row, result_col)
GROUP BY result_row;

 * sqlite:///hw1.db
Done.


C
"106 , 80 , 171"
"146 , 109 , 212"
"23 , 17 , 55"


### Part (d): Matrix power [8 points]

The power $A^n$ of a matrix $A$ is defined as the matrix product of $n$ copies of $A$. 

Write a _single SQL query_ that computes the **third power** of matrix $A$, in other words, $A^3 = A \cdot A \cdot A$:

In [9]:
%%sql
SELECT group_concat(val, ' , ') AS "C" FROM

(SELECT
    A.i AS i,
    A3.j AS j,
    SUM(A.val * A3.val) AS val
FROM A JOIN 
(SELECT
    A.i AS i,
    A2.j AS j,
    SUM(A.val * A2.val) AS val
FROM A JOIN A AS "A2" ON A.j = A2.i
GROUP BY A.i, A2.j
ORDER BY i, j) AS "A3" ON A.j = A3.i
GROUP BY A.i, A3.j
ORDER BY i, j)

GROUP BY i;

 * sqlite:///hw1.db
Done.


C
"1767 , 1065 , 2065"
"2396 , 1463 , 2745"
"350 , 190 , 467"


Problem 3: The Sales Database [25 points]
----------------------------------------------

We've prepared and loaded a dataset related to sales data from a company. The dataset has the following schema:

> `Holidays (WeekDate, IsHoliday)`

> `Stores (Store, Type, Size)`

> `TemporalData(Store, WeekDate, Temperature, FuelPrice, CPI, UnemploymentRate)`

> `Sales (Store, Dept, WeekDate, WeeklySales)`

Before you start writing queries on the database, find the schema and the constraints (keys, foreign keys). 

### Part (a): Sales during Holidays [8 points]

Using a _single SQL query_, find the store(s) with the largest overall sales during holiday weeks. Further requirements:
* Use the `WITH` clause before the main body of the query to compute a subquery if necessary.
* Return a relation with schema `(Store, AllSales)`.

Write your query here:

In [10]:
%%sql

WITH HolidaySales AS (
    SELECT Sales.Store, SUM(WeeklySales) AS AllSales
    FROM Sales JOIN Holidays ON Sales.WeekDate = Holidays.WeekDate JOIN Stores ON Sales.Store = Stores.Store
    WHERE Holidays.IsHoliday = 'TRUE'
    GROUP BY Sales.Store
)

SELECT Store, AllSales
FROM HolidaySales
WHERE AllSales = (SELECT MAX(AllSales) FROM HolidaySales)


 * sqlite:///hw1.db
Done.


Store,AllSales
20,22490350.81


### Part (b): When Holidays do not help Sales [9 points]

Using a _single SQL query_, compute the **number** of non-holiday weeks that had larger sales than the overall average sales during holiday weeks. Further requirements:
* Use the `WITH` clause before the main body of the query to compute a subquery if necessary.
* Return a relation with schema `(NumNonHolidays)`.

Write your query here:

In [11]:
%%sql
WITH OverallWeeklySales AS (
    SELECT Sales.WeekDate, IsHoliday, SUM(WeeklySales) AS WeeklySales
    FROM Sales JOIN Holidays ON Sales.WeekDate = Holidays.WeekDate
    GROUP BY Sales.WeekDate
)

SELECT COUNT(WeekDate) AS NumNonHolidays FROM OverallWeeklySales
WHERE IsHoliday = 'FALSE' AND WeeklySales > (SELECT AVG(WeeklySales) AS AvgHolidaySales
FROM OverallWeeklySales
WHERE IsHoliday = 'TRUE')

 * sqlite:///hw1.db
Done.


NumNonHolidays
8


### Part (c): Total Summer Sales [8 points]

Using a _single SQL query_, compute the total sales during summer (months 6,7,and 8) for each type of store. Further requirements:
* Return a relation with schema `(type, TotalSales)`.

*Hint:* SQLite3 does not support native operations on the DATE datatype. To create a workaround, you can use the `LIKE` predicate and the string concatenation operator (||). You can also use the substring operator that SQLite3 supports (`substr`).

Write your query here:

> `Holidays (WeekDate, IsHoliday)`

> `Stores (Store, Type, Size)`

> `TemporalData(Store, WeekDate, Temperature, FuelPrice, CPI, UnemploymentRate)`

> `Sales (Store, Dept, WeekDate, WeeklySales)`

In [31]:
%%sql
SELECT type, SUM(WeeklySales) AS TotalSales
FROM Sales JOIN Stores ON Stores.Store = Sales.Store
WHERE WeekDate LIKE '%-06-%' OR WeekDate LIKE '%-07-%' OR WeekDate LIKE '%-08-%'
GROUP BY type

 * sqlite:///hw1.db
Done.


type,TotalSales
A,1211554899.85
B,561610722.4
C,112555450.66


Problem 4: The Traveling SQL Server Salesman Problem [25 points]
--------------------------------------------------

SQL Server salespeople are lucky as far as traveling salespeople go- they only have to sell one or two big enterprise contracts, at one or two offices in Wisconsin, in order to make their monthly quota!

Answer the following questions using the table of streets connecting company office buildings.

**Note that for convenience all streets are included _twice_, as $A \rightarrow B$ and $B \rightarrow A$.  This should make some parts of the problem easier, but remember to take it into account!**

In [13]:
%sql SELECT * FROM streets LIMIT 4;

 * sqlite:///hw1.db
Done.


id,direction,A,B,d
0,F,UW-Madison,DooHickey Collective,7
0,R,DooHickey Collective,UW-Madison,7
1,F,DooHickey Collective,Gizmo Corp,2
1,R,Gizmo Corp,DooHickey Collective,2


### Part (a): One-hop, two-hop, three-hop... [9 points]

Our salesperson has stopped at UW-Madison, to steal some cool new RDBMS technology from CS564-ers, and now wants to go sell it to a company _within 9 miles of UW-Madison and _passing through no more than 3 distinct streets_.  Write a single query, not using `WITH` (see later on), to find all such companies.

Your query should return the schema `(company, distance)` where distance is cumulative from UW-Madison.

Write your query here:

In [ ]:
%%sql
SELECT company, MIN(distance) AS "distance"
FROM
(SELECT B AS "company", d AS "distance"
FROM streets
WHERE A = 'UW-Madison' AND d <= 9

UNION

SELECT hop2.B AS "company", hop1.d + hop2.d AS "distance"
FROM streets AS "hop1" JOIN streets AS "hop2" ON hop1.B = hop2.A
WHERE hop1.A = "UW-Madison" AND hop1.direction = "F" AND hop2.direction = "F" AND hop1.d + hop2.d <=9 

UNION

SELECT hop3.B AS "company", hop1.d + hop2.d + hop3.d AS "distance"
FROM streets AS "hop1" JOIN streets AS "hop2" ON hop1.B = hop2.A JOIN streets AS "hop3" ON hop2.B = hop3.A
WHERE hop1.A = "UW-Madison" AND hop1.direction = "F" AND hop2.direction = "F" AND hop3.direction = "F" AND hop3.B != "UW-Madison" AND hop1.d + hop2.d + hop3.d <=9)
GROUP BY company

 * sqlite:///hw1.db
Done.


company,distance
DooHickey Collective,7
DooHickey Corp,9
Gadget Collective,9
Gadget Corp,6
Gizmo Corp,9


### Part (b): A stop at the Farm [8 points]

Now, our salesperson is out in the field, and wants to see all routes- and their distances- which will take him/her from a company $A$ to a company $B$, with the following constraints:
* The route must pass through UW-Madison (in order to pick up new RDBMS tech to sell!)
* $A$ and $B$ must _each individually_ be within 2 hops of UW-Madison
* $A$ and $B$ must be different companies
* _The total distance must be $<= 15$_
* Do not use `WITH`
* If you return a path $A \rightarrow B$, _do not include_ $B \rightarrow A$ in your answer!

In order to make your answer a bit cleaner, you may split into two queries, one of which creates a `VIEW`.  A view is a virtual table based on the output set of a SQL query.  A view can be used just like a normal table- the only difference under the hood is that the DBMS re-evaluates the query used to generate it each time a view is queried by a user (thus the data is always up-to date!)

Here's a simple example of a view:

In [15]:
%%sql 
DROP VIEW IF EXISTS short_streets;
CREATE VIEW short_streets AS 
SELECT A, B, d FROM streets WHERE d < 3;
SELECT * FROM short_streets LIMIT 3;

 * sqlite:///hw1.db
Done.
Done.
Done.


A,B,d
DooHickey Collective,Gizmo Corp,2
Gizmo Corp,DooHickey Collective,2
Gizmo Corp,Widget Industries,1


Write your query or queries here:

In [88]:
%%sql
DROP VIEW IF EXISTS UWToB;
CREATE VIEW UWToB AS
SELECT A, B, MIN(d) AS "d"
FROM
(SELECT A, B, d
FROM streets
WHERE A = 'UW-Madison'

UNION

SELECT hop1.A, hop2.B, hop1.d + hop2.d
FROM streets AS "hop1" JOIN streets AS "hop2" ON hop1.B = hop2.A
WHERE hop1.A = "UW-Madison" AND hop1.direction = "F" AND hop2.direction = "F")
GROUP BY A, B;


SELECT A, B, TotalDistance
FROM
(SELECT AToUW.A, UWToB.B, MIN(AToUW.A, UWToB.B) || "-" || MAX(AToUW.A, UWToB.B) AS "path", AToUW.d + UWToB.d AS "TotalDistance"
FROM (SELECT A, B, d
FROM streets
WHERE B = 'UW-Madison'

UNION

SELECT hop1.A, hop2.B, hop1.d + hop2.d
FROM streets AS "hop1" JOIN streets AS "hop2" ON hop1.B = hop2.A
WHERE hop2.B = "UW-Madison" AND hop1.direction = "F" AND hop2.direction = "F") AS AToUW JOIN UWToB ON AToUW.B = UWToB.A
WHERE AToUW.A != UWToB.B AND TotalDistance <= 15
GROUP BY path)

 * sqlite:///hw1.db
Done.
Done.
Done.


A,B,TotalDistance
Gadget Corp,DooHickey Collective,13
Gadget Corp,DooHickey Corp,15
Gadget Corp,Gadget Collective,15
Gadget Corp,Gizmo Corp,15


### Part (c): Finding Triangles [8 points]

Finally, our salesperson wants to find a route that goes from company $A$ to company $B$ to company $C$ and then back to company $A$ with the following constraints:
* $A$, $B$, $C$ must be different companies
* Do not use `WITH` 
* Output each such route that you find once (use the id's as a way to break ties)
* Output the distance of the route

Write your query here:

In [112]:
%%sql
SELECT hop1.A, hop2.A AS "B", hop3.A AS "C", hop1.d + hop2.d + hop3.d as "TotalDistance"
FROM streets AS "hop1" JOIN streets AS "hop2" ON hop1.B = hop2.A JOIN streets AS "hop3" ON hop2.B = hop3.A
WHERE hop3.B = hop1.A AND hop1.id < hop2.id AND hop1.id < hop3.id

 * sqlite:///hw1.db
Done.


A,B,C,TotalDistance
Thing Industries,Widget Collective,GadgetCo,18
Widget Collective,Thing Industries,GadgetCo,18
